# LeetCode 993 — Cousins in Binary Tree bằng Python

**Link đề:** https://leetcode.com/problems/cousins-in-binary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isCousins(self, root: TreeNode | None, x: int, y: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu hai khái niệm trên cây: **độ sâu** (depth) và **cha** (parent) của một node.
2. Viết DFS đệ quy **mang theo thông tin xuống dưới** (tham số `depth`, `parent`).
3. Viết BFS **theo từng tầng** với `collections.deque` — công cụ tự nhiên cho câu hỏi "cùng tầng".
4. So sánh 4 cách: DFS ghi nhận, BFS kèm cha, BFS kiểm tra anh em, DFS dừng sớm.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Cho `root` của cây nhị phân có **giá trị các node đôi một khác nhau**, và hai giá trị **khác nhau** `x`, `y` (chắc chắn có trong cây). Trả về `True` nếu hai node mang `x` và `y` là **anh em họ** (cousins).

Hai node là **cousins** nếu: **cùng độ sâu** **và** **khác cha**. Root có độ sâu 0; con của node độ sâu `k` có độ sâu `k + 1`.

```text
Ví dụ 1: [1,2,3,4], x=4, y=3   -> False
        1            depth(4)=2, depth(3)=1  -> khác tầng
       / \
      2   3
     /
    4

Ví dụ 2: [1,2,3,null,4,null,5], x=5, y=4  -> True
        1            depth đều = 2; cha của 4 là 2, cha của 5 là 3
       / \
      2   3
       \   \
        4   5

Ví dụ 3: [1,2,3,null,4], x=2, y=3  -> False
        1            cùng tầng 1 nhưng CÙNG CHA (1) -> là anh em ruột, không phải họ
       / \
      2   3
       \
        4
```

**Constraints:** số node trong `[2, 100]`; `1 <= Node.val <= 100`; giá trị duy nhất; `x != y`; `x`, `y` có trong cây.

## 2. Nền tảng: cây nhị phân trong Python

### 2.1. `TreeNode` và hàm dựng cây từ list LeetCode

Để notebook chạy độc lập, ta tự định nghĩa `TreeNode` và hàm `build_tree` nhận list kiểu level-order (`None` = vị trí trống).

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: list):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root

def show(node, depth=0):
    # In cây xoay ngang: con phải ở trên, con trái ở dưới
    if node is None:
        return
    show(node.right, depth + 1)
    print("    " * depth + str(node.val))
    show(node.left, depth + 1)

t2 = build_tree([1, 2, 3, None, 4, None, 5])
show(t2)

        5
    3
1
        4
    2


### 2.2. Độ sâu và cha — tính bằng DFS mang tham số

Khi đi từ cha xuống con, độ sâu tăng 1 và **cha của con chính là node hiện tại**. Ta truyền hai thông tin này xuống qua tham số.

In [2]:
def list_depth_parent(node, depth=0, parent=None):
    if node is None:
        return
    p = parent.val if parent else None
    print(f"node {node.val}: depth={depth}, parent={p}")
    list_depth_parent(node.left, depth + 1, node)
    list_depth_parent(node.right, depth + 1, node)

list_depth_parent(t2)

node 1: depth=0, parent=None
node 2: depth=1, parent=1
node 4: depth=2, parent=2
node 3: depth=1, parent=1
node 5: depth=2, parent=3


### 2.3. `deque` và BFS theo tầng

BFS dùng **hàng đợi**: lấy node ở đầu (`popleft`), đưa các con vào cuối (`append`). Muốn xử lý **trọn một tầng** mỗi lần, ta ghi nhớ `len(queue)` **trước** khi xử lý — đó đúng là số node của tầng hiện tại.

In [3]:
def print_levels(root):
    queue = deque([root])
    depth = 0
    while queue:
        size = len(queue)                 # số node của tầng này
        level = []
        for _ in range(size):
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        print(f"tầng {depth}: {level}")
        depth += 1

print_levels(t2)

tầng 0: [1]
tầng 1: [2, 3]
tầng 2: [4, 5]


`for _ in range(size)`: dấu gạch dưới `_` là quy ước cho biến **không dùng tới** — ta chỉ cần lặp đúng `size` lần.

## 3. Cách trực giác (baseline) — ghi nhận depth & parent của MỌI node

Duyệt toàn bộ cây, lưu `info[val] = (depth, parent_val)` vào một **dict**. Sau đó tra `x`, `y` và so sánh.

In [4]:
def is_cousins_map(root, x: int, y: int) -> bool:
    info = {}
    def dfs(node, depth, parent):
        if node is None:
            return
        info[node.val] = (depth, parent)
        dfs(node.left, depth + 1, node.val)
        dfs(node.right, depth + 1, node.val)
    dfs(root, 0, None)
    dx, px = info[x]
    dy, py = info[y]
    return dx == dy and px != py

print(is_cousins_map(build_tree([1, 2, 3, 4]), 4, 3))
print(is_cousins_map(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4))
print(is_cousins_map(build_tree([1, 2, 3, None, 4]), 2, 3))

False
True
False


`dx, px = info[x]` là **unpacking** một tuple 2 phần tử vào 2 biến.

Độ phức tạp: O(n) thời gian, O(n) bộ nhớ cho dict. Hoàn toàn chấp nhận được với `n ≤ 100`, nhưng ta lưu thông tin của **mọi** node trong khi chỉ cần **hai** node.

## 4. Nút thắt

- Lưu thừa: chỉ cần (depth, parent) của `x` và `y`.
- Duyệt thừa: nếu đã tìm thấy cả `x` và `y` thì có thể **dừng**. Thậm chí, nếu BFS đã xong một tầng mà chỉ thấy **một** trong hai → chắc chắn **không** cùng tầng → trả `False` ngay.

## 5. Insight

Cousins = **cùng tầng** + **khác cha**. Hai cách nhìn:

1. **DFS:** chỉ ghi (depth, parent) khi gặp `x` hoặc `y`.
2. **BFS theo tầng:** trong mỗi tầng, kiểm tra xem có thấy cả `x` và `y` không. Điều kiện "khác cha" ⇔ chúng **không phải hai con của cùng một node**. Ta có thể kiểm tra trực tiếp khi đứng ở node cha: nếu `{node.left.val, node.right.val} == {x, y}` thì là anh em ruột → `False`.

In [5]:
# Kiểm tra "anh em ruột" khi đứng ở node cha
node = build_tree([1, 2, 3])
children = {node.left.val, node.right.val}
print("con của 1:", children, "| {2, 3} là anh em ruột?", children == {2, 3})

con của 1: {2, 3} | {2, 3} là anh em ruột? True


## 6. Lời giải chính (BFS theo tầng) — xây từng đoạn rồi ghép

**Đoạn 1:** `queue = deque([root])`, lặp khi còn node.

**Đoạn 2:** với mỗi tầng, đặt `found_x = found_y = False`, duyệt đúng `len(queue)` node.

**Đoạn 3:** với mỗi node:
- Nếu hai con của nó là đúng `x` và `y` → anh em ruột → `return False`.
- Đánh dấu `found_x`/`found_y` nếu `node.val` là `x`/`y`.
- Đưa các con vào queue.

**Đoạn 4:** cuối tầng:
- Thấy cả hai → `return True` (cùng tầng, và đã loại trường hợp cùng cha).
- Thấy đúng một → `return False` (khác tầng).

In [6]:
def is_cousins_bfs(root, x: int, y: int) -> bool:
    queue = deque([root])                                  # Đoạn 1
    while queue:
        found_x = found_y = False                          # Đoạn 2
        for _ in range(len(queue)):
            node = queue.popleft()
            if node.left and node.right and {node.left.val, node.right.val} == {x, y}:
                return False                               # Đoạn 3: anh em ruột
            if node.val == x:
                found_x = True
            if node.val == y:
                found_y = True
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        if found_x and found_y:                            # Đoạn 4
            return True
        if found_x or found_y:
            return False
    return False

print(is_cousins_bfs(build_tree([1, 2, 3, 4]), 4, 3))
print(is_cousins_bfs(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4))
print(is_cousins_bfs(build_tree([1, 2, 3, None, 4]), 2, 3))

False
True
False


`found_x = found_y = False` là **gán dây chuyền**: cả hai biến cùng nhận `False`.

## 7. Trace từng bước

Ví dụ 2: `[1,2,3,null,4,null,5]`, `x = 5`, `y = 4`.

| Tầng | Node xét | Hai con | Anh em ruột {5,4}? | found_x (5) | found_y (4) | Cuối tầng |
|:---:|:---:|---|:---:|:---:|:---:|---|
| 0 | 1 | {2, 3} | không | F | F | chưa thấy ai → tiếp |
| 1 | 2 | chỉ có 4 | không | F | F | |
| 1 | 3 | chỉ có 5 | không | F | F | chưa thấy ai → tiếp |
| 2 | 4 | — | — | F | **T** | |
| 2 | 5 | — | — | **T** | T | thấy cả hai → **True** |

In [7]:
def trace_cousins(root, x, y):
    print(f"x={x}, y={y}")
    queue = deque([root])
    depth = 0
    while queue:
        found_x = found_y = False
        print(f"Tầng {depth}: queue = {[n.val for n in queue]}")
        for _ in range(len(queue)):
            node = queue.popleft()
            kids = [c.val for c in (node.left, node.right) if c]
            sib = node.left and node.right and {node.left.val, node.right.val} == {x, y}
            print(f"   node {node.val}: con = {kids}, anh em ruột x,y? {bool(sib)}")
            if sib:
                print("   => x, y cùng cha -> False")
                return False
            found_x |= node.val == x
            found_y |= node.val == y
            for c in (node.left, node.right):
                if c:
                    queue.append(c)
        print(f"   cuối tầng: found_x={found_x}, found_y={found_y}")
        if found_x and found_y:
            print("   => cùng tầng, khác cha -> True")
            return True
        if found_x or found_y:
            print("   => chỉ thấy một -> khác tầng -> False")
            return False
        depth += 1
    return False

trace_cousins(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4)
print()
trace_cousins(build_tree([1, 2, 3, 4]), 4, 3)
print()
trace_cousins(build_tree([1, 2, 3, None, 4]), 2, 3)

x=5, y=4
Tầng 0: queue = [1]
   node 1: con = [2, 3], anh em ruột x,y? False
   cuối tầng: found_x=False, found_y=False
Tầng 1: queue = [2, 3]
   node 2: con = [4], anh em ruột x,y? False
   node 3: con = [5], anh em ruột x,y? False
   cuối tầng: found_x=False, found_y=False
Tầng 2: queue = [4, 5]
   node 4: con = [], anh em ruột x,y? False
   node 5: con = [], anh em ruột x,y? False
   cuối tầng: found_x=True, found_y=True
   => cùng tầng, khác cha -> True

x=4, y=3
Tầng 0: queue = [1]
   node 1: con = [2, 3], anh em ruột x,y? False
   cuối tầng: found_x=False, found_y=False
Tầng 1: queue = [2, 3]
   node 2: con = [4], anh em ruột x,y? False
   node 3: con = [], anh em ruột x,y? False
   cuối tầng: found_x=False, found_y=True
   => chỉ thấy một -> khác tầng -> False

x=2, y=3
Tầng 0: queue = [1]
   node 1: con = [2, 3], anh em ruột x,y? True
   => x, y cùng cha -> False


False

`found_x |= cond` là viết tắt của `found_x = found_x | cond` — với giá trị bool, `|` hoạt động như "hoặc".

## 8. Code nộp LeetCode

### Cách A — BFS theo tầng, kiểm tra anh em ruột ở node cha

In [8]:
class Solution:
    def isCousins(self, root: TreeNode | None, x: int, y: int) -> bool:
        queue = deque([root])
        while queue:
            found_x = found_y = False
            for _ in range(len(queue)):
                node = queue.popleft()
                if node.left and node.right and {node.left.val, node.right.val} == {x, y}:
                    return False
                if node.val == x:
                    found_x = True
                elif node.val == y:
                    found_y = True
                if node.left:
                    queue.append(node.left)
                if node.right:
                    queue.append(node.right)
            if found_x and found_y:
                return True
            if found_x or found_y:
                return False
        return False

s = Solution()
print(s.isCousins(build_tree([1, 2, 3, 4]), 4, 3))
print(s.isCousins(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4))
print(s.isCousins(build_tree([1, 2, 3, None, 4]), 2, 3))

False
True
False


### Cách B — DFS chỉ ghi nhận `x`, `y`, dừng sớm khi đủ

In [9]:
class SolutionDFS:
    def isCousins(self, root: TreeNode | None, x: int, y: int) -> bool:
        found = {}                                   # val -> (depth, parent_val)

        def dfs(node, depth, parent):
            if node is None or len(found) == 2:      # dừng sớm khi đã đủ
                return
            if node.val in (x, y):
                found[node.val] = (depth, parent)
            dfs(node.left, depth + 1, node.val)
            dfs(node.right, depth + 1, node.val)

        dfs(root, 0, None)
        (dx, px), (dy, py) = found[x], found[y]
        return dx == dy and px != py

print(SolutionDFS().isCousins(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4))

True


`(dx, px), (dy, py) = found[x], found[y]` — **unpacking lồng nhau**: vế phải là 2 tuple, mỗi tuple tách tiếp thành 2 biến.

### Cách C — BFS lưu cặp (node, parent) trong queue

Thay vì kiểm tra anh em ruột ở node cha, ta đưa **cha** đi kèm vào queue, và ghi lại cha của `x`, `y` khi gặp trong cùng một tầng.

In [10]:
class SolutionBFSParent:
    def isCousins(self, root: TreeNode | None, x: int, y: int) -> bool:
        queue = deque([(root, None)])
        while queue:
            parent_x = parent_y = None
            for _ in range(len(queue)):
                node, parent = queue.popleft()
                if node.val == x:
                    parent_x = parent
                if node.val == y:
                    parent_y = parent
                for child in (node.left, node.right):
                    if child:
                        queue.append((child, node))
            if parent_x and parent_y:
                return parent_x is not parent_y
            if parent_x or parent_y:
                return False
        return False

print(SolutionBFSParent().isCousins(build_tree([1, 2, 3, None, 4]), 2, 3))
print(SolutionBFSParent().isCousins(build_tree([1, 2, 3, None, 4, None, 5]), 5, 4))

False
True


`is not` so sánh **danh tính object** (có phải cùng một node không) — đúng ý "cùng một cha" hơn là so `val`.

Lưu ý tinh tế: ở đây `parent_x` vừa là "cha của x" vừa là **cờ** "đã thấy x". Nếu `x` là root thì cha là `None`, nên ở tầng 0 ta coi như "chưa thấy". Kết quả vẫn đúng: `y` nằm ở tầng sâu hơn và sẽ được thấy **một mình** → `False`, đúng vì root không có anh em họ.

### Cách D — một DFS trả về depth và parent của một giá trị

In [11]:
class SolutionFind:
    def isCousins(self, root: TreeNode | None, x: int, y: int) -> bool:
        def find(node, target, depth, parent):
            if node is None:
                return None
            if node.val == target:
                return depth, parent
            return (find(node.left, target, depth + 1, node)
                    or find(node.right, target, depth + 1, node))

        dx, px = find(root, x, 0, None)
        dy, py = find(root, y, 0, None)
        return dx == dy and px is not py

print(SolutionFind().isCousins(build_tree([1, 2, 3, 4]), 4, 3))

False


`A or B`: nếu `A` "truthy" (ở đây là tuple khác rỗng) thì trả `A`, ngược lại trả `B`. Nhờ đó tìm thấy ở nhánh trái thì không tìm nhánh phải.

## 9. Vì sao đúng?

**Cách A (BFS).** Invariant: ở đầu mỗi vòng `while`, `queue` chứa **đúng tất cả** node của tầng `d` (theo thứ tự trái → phải). Vòng `for` lặp đúng `len(queue)` lần nên chỉ xử lý tầng `d`, các con được thêm vào sau tạo thành tầng `d + 1`.
- Nếu `x`, `y` đều ở tầng `d`: hoặc chúng cùng cha — khi đó cha nằm ở tầng `d − 1` và đã bị phát hiện ở vòng trước (`return False`), hoặc khác cha → cuối tầng thấy cả hai → `True`. ✔
- Nếu khác tầng: ở tầng nông hơn ta thấy đúng một → `False`. ✔
- Mọi node được xét đúng một lần nên không bỏ sót.

**Cách B/D (DFS).** Tham số `depth` và `parent` khi thăm node `v` đúng bằng độ sâu và cha của `v` (quy nạp: root có `0, None`; con nhận `depth + 1` và cha là node hiện tại). Vì giá trị duy nhất nên ghi nhận theo `val` không bị nhầm.

## 10. Độ phức tạp

Gọi `n` = số node, `h` = chiều cao, `w` = số node lớn nhất trong một tầng.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Baseline dict mọi node | O(n) | O(n) | dễ nhất |
| A. BFS + kiểm tra anh em | O(n) | O(w) | dừng ngay khi xong tầng chứa x hoặc y |
| B. DFS ghi x, y, dừng sớm | O(n) | O(h) | đệ quy |
| C. BFS kèm cha | O(n) | O(w) | không cần kiểm tra anh em riêng |
| D. Hai lần tìm | O(n) (2 lượt) | O(h) | tách bạch, dễ đọc |

## 11. Bộ test

In [12]:
tests = [
    ([1, 2, 3, 4], 4, 3, False),
    ([1, 2, 3, None, 4, None, 5], 5, 4, True),
    ([1, 2, 3, None, 4], 2, 3, False),            # anh em ruột
    ([1, 2], 1, 2, False),                         # x là root
    ([1, 2, 3, 4, 5, 6, 7], 4, 6, True),
    ([1, 2, 3, 4, 5, 6, 7], 4, 5, False),          # cùng cha 2
    ([1, 2, 3, 4, 5, 6, 7], 2, 7, False),          # khác tầng
    ([1, 2, 3, None, 4, 5, None, 6, None, None, 7], 6, 7, True),  # sâu 3, cha 4 và 5
]
impls = {
    "dict": is_cousins_map, "bfs": is_cousins_bfs,
    "LC-A": Solution().isCousins, "LC-B": SolutionDFS().isCousins,
    "LC-C": SolutionBFSParent().isCousins, "LC-D": SolutionFind().isCousins,
}
for values, x, y, expected in tests:
    root = build_tree(values)
    for name, f in impls.items():
        got = f(root, x, y)
        assert got == expected, (name, values, x, y, got, expected)
        assert f(root, y, x) == expected, (name, "đổi chỗ x,y")
    print(f"{str(values):45} x={x}, y={y} -> {expected} PASS")
print("Tất cả test PASS (kể cả khi đổi chỗ x và y)!")

[1, 2, 3, 4]                                  x=4, y=3 -> False PASS
[1, 2, 3, None, 4, None, 5]                   x=5, y=4 -> True PASS
[1, 2, 3, None, 4]                            x=2, y=3 -> False PASS
[1, 2]                                        x=1, y=2 -> False PASS
[1, 2, 3, 4, 5, 6, 7]                         x=4, y=6 -> True PASS
[1, 2, 3, 4, 5, 6, 7]                         x=4, y=5 -> False PASS
[1, 2, 3, 4, 5, 6, 7]                         x=2, y=7 -> False PASS
[1, 2, 3, None, 4, 5, None, 6, None, None, 7] x=6, y=7 -> True PASS
Tất cả test PASS (kể cả khi đổi chỗ x và y)!


In [13]:
import random

def random_tree(rng, n):
    vals = rng.sample(range(1, 101), n)
    root = TreeNode(vals[0])
    nodes = [root]
    for v in vals[1:]:
        while True:
            parent = rng.choice(nodes)
            side = rng.choice(["left", "right"])
            if getattr(parent, side) is None:
                child = TreeNode(v)
                setattr(parent, side, child)
                nodes.append(child)
                break
    return root, vals

rng = random.Random(993)
for _ in range(2000):
    root, vals = random_tree(rng, rng.randint(2, 15))
    x, y = rng.sample(vals, 2)
    expected = is_cousins_map(root, x, y)
    for name, f in impls.items():
        assert f(root, x, y) == expected, (name, vals, x, y)
print("2000 cây ngẫu nhiên: mọi cách khớp")

2000 cây ngẫu nhiên: mọi cách khớp


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra cùng độ sâu, quên điều kiện khác cha

In [14]:
def wrong_depth_only(root, x, y):
    info = {}
    def dfs(node, d):
        if node:
            info[node.val] = d
            dfs(node.left, d + 1); dfs(node.right, d + 1)
    dfs(root, 0)
    return info[x] == info[y]

t = build_tree([1, 2, 3, None, 4])
print("Sai :", wrong_depth_only(t, 2, 3))
print("Đúng:", is_cousins_bfs(t, 2, 3))

Sai : True
Đúng: False


**Vì sao sai:** 2 và 3 cùng cha (anh em ruột), không phải anh em họ. **Sửa:** so cả `parent`.

### Lỗi 2: BFS không tách tầng

In [15]:
def wrong_no_levels(root, x, y):
    queue = deque([(root, 0, None)])
    seen = {}
    while queue:
        node, d, p = queue.popleft()
        if node.val in (x, y):
            seen[node.val] = p
            if len(seen) == 2:
                return seen[x] is not seen[y]     # quên so độ sâu!
        for c in (node.left, node.right):
            if c:
                queue.append((c, d + 1, node))
    return False

t = build_tree([1, 2, 3, 4])
print("Sai :", wrong_no_levels(t, 4, 3))
print("Đúng:", is_cousins_bfs(t, 4, 3))

Sai : True
Đúng: False


**Vì sao sai:** gặp đủ hai node nhưng chúng ở **khác tầng**. **Sửa:** xử lý từng tầng bằng `for _ in range(len(queue))`, hoặc lưu và so cả độ sâu.

### Lỗi 3: Dùng `len(queue)` thay đổi trong vòng lặp

Viết `while i < len(queue)` trong khi vẫn `append` con vào → tầng hiện tại "dài ra" và trộn với tầng sau. Phải chốt `size = len(queue)` **trước** vòng `for` (`range(len(queue))` đã tính một lần duy nhất).

### Lỗi 4: Kiểm tra anh em ruột mà không kiểm tra con tồn tại

`{node.left.val, node.right.val}` khi `node.left` là `None` → `AttributeError`. Luôn kiểm tra `node.left and node.right` trước.

### Lỗi 5: Hiểu nhầm độ sâu bắt đầu từ 1

Không ảnh hưởng kết quả (chỉ so sánh tương đối), nhưng khi in/ghi chú dễ gây nhầm lẫn — đề quy ước root có depth 0.

## 13. Mẹo nhận dạng pattern

- Câu hỏi về **"cùng tầng"**, "theo từng tầng", "tầng nào" → **BFS theo tầng** với `for _ in range(len(queue))`.
- Cần **quan hệ cha–con** → truyền `parent` xuống trong DFS, hoặc đưa `(node, parent)` vào queue, hoặc kiểm tra ở node cha.
- Cần **độ sâu** → truyền `depth + 1` xuống khi đệ quy.
- Giá trị node **duy nhất** → dùng `val` làm khóa dict an toàn.

## 14. Biến thể

1. **LeetCode 2641 — Cousins in Binary Tree II:** thay giá trị mỗi node bằng **tổng giá trị các anh em họ** → BFS theo tầng, tổng tầng trừ tổng của các anh em ruột.
2. **LeetCode 102 — Binary Tree Level Order Traversal:** in từng tầng.
3. **LeetCode 104 — Maximum Depth:** độ sâu lớn nhất.
4. **LeetCode 236 — Lowest Common Ancestor:** tổ tiên chung thấp nhất của hai node.

In [16]:
def replace_with_cousin_sum(root):
    # Biến thể 1: mỗi node := tổng các node cùng tầng trừ anh em ruột (kể cả chính nó)
    root.val = 0
    level = [root]
    while level:
        next_level = []
        total = sum(c.val for node in level for c in (node.left, node.right) if c)
        for node in level:
            sib_sum = sum(c.val for c in (node.left, node.right) if c)
            for c in (node.left, node.right):
                if c:
                    c.val = total - sib_sum
                    next_level.append(c)
        level = next_level
    return root

def to_list(root):
    out, q = [], deque([root])
    while q:
        n = q.popleft()
        out.append(n.val if n else None)
        if n:
            q.append(n.left); q.append(n.right)
    while out and out[-1] is None:
        out.pop()
    return out

print(to_list(replace_with_cousin_sum(build_tree([5, 4, 9, 1, 10, None, 7]))))

[0, 0, 0, 7, 7, None, 11]


Kết quả khớp ví dụ của LeetCode 2641: `[0,0,0,7,7,null,11]`. Biểu thức `sum(c.val for node in level for c in (...) if c)` là generator **hai vòng `for` lồng** — đọc từ trái sang phải như hai vòng lặp lồng nhau.

## 15. Bài tập tự luyện (TODO)

### Bài 1 — Độ sâu của một giá trị
Trả về độ sâu của node mang giá trị `target` (root = 0), hoặc `-1` nếu không có.

In [17]:
def depth_of(root, target: int) -> int:
    # TODO
    pass

# assert depth_of(build_tree([1, 2, 3, None, 4]), 4) == 2
# assert depth_of(build_tree([1, 2, 3]), 9) == -1

### Bài 2 — Cha của một giá trị
Trả về **giá trị** của cha node `target`, hoặc `None` nếu `target` là root.

In [18]:
def parent_of(root, target: int):
    # TODO
    pass

# assert parent_of(build_tree([1, 2, 3, None, 4]), 4) == 2
# assert parent_of(build_tree([1, 2, 3]), 1) is None

### Bài 3 — Level order
Trả về list các tầng, mỗi tầng là list giá trị (LeetCode 102).

In [19]:
def level_order(root) -> list[list[int]]:
    # TODO
    pass

# assert level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]

### Bài 4 — Liệt kê anh em họ
Trả về list (đã sắp xếp) các giá trị là **anh em họ** của `target`.

In [20]:
def cousins_of(root, target: int) -> list[int]:
    # TODO
    pass

# assert cousins_of(build_tree([1, 2, 3, 4, 5, 6, 7]), 4) == [6, 7]
# assert cousins_of(build_tree([1, 2, 3]), 2) == []

### Bài 5 — Anh em ruột
Trả về `True` nếu `x` và `y` là anh em **ruột** (cùng cha).

In [21]:
def are_siblings(root, x: int, y: int) -> bool:
    # TODO
    pass

# assert are_siblings(build_tree([1, 2, 3, None, 4]), 2, 3) == True
# assert are_siblings(build_tree([1, 2, 3, 4, 5, 6, 7]), 4, 6) == False

### Bài 6 (nâng cao) — Khoảng cách giữa hai node
Trả về số cạnh trên đường đi giữa `x` và `y` (gợi ý: tổ tiên chung thấp nhất).

In [22]:
def distance(root, x: int, y: int) -> int:
    # TODO
    pass

# assert distance(build_tree([1, 2, 3, 4, 5, 6, 7]), 4, 6) == 4
# assert distance(build_tree([1, 2, 3, 4, 5, 6, 7]), 4, 5) == 2
# assert distance(build_tree([1, 2, 3, 4, 5, 6, 7]), 1, 7) == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | DFS mang `depth` | tìm thấy thì trả về ngay | trả `-1` khi `None`; lấy nhánh nào khác `-1` |
| 2 | DFS mang `parent` | so `node.val == target` | trả `parent.val if parent else None` |
| 3 | BFS theo tầng | `size = len(queue)` | gom `level` rồi `append` vào kết quả |
| 4 | tìm tầng của target | trong tầng đó, bỏ con của cha target | dùng BFS kèm cha |
| 5 | kiểm tra ở node cha | `{left.val, right.val} == {x, y}` | duyệt mọi node |
| 6 | LCA | `dist = depth(x) + depth(y) − 2·depth(lca)` | LCA đệ quy: trả node nếu là x/y, nếu cả 2 nhánh khác None thì node là LCA |

## 17. Đáp án tham khảo

In [23]:
def ans_depth_of(root, target, d=0):
    if root is None:
        return -1
    if root.val == target:
        return d
    left = ans_depth_of(root.left, target, d + 1)
    return left if left != -1 else ans_depth_of(root.right, target, d + 1)

def ans_parent_of(root, target):
    def dfs(node, parent):
        if node is None:
            return "missing"
        if node.val == target:
            return parent.val if parent else None
        r = dfs(node.left, node)
        return r if r != "missing" else dfs(node.right, node)
    return dfs(root, None)

def ans_level_order(root):
    res, queue = [], deque([root] if root else [])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.val)
            for c in (node.left, node.right):
                if c:
                    queue.append(c)
        res.append(level)
    return res

def ans_cousins_of(root, target):
    queue = deque([(root, None)])
    while queue:
        level = [queue.popleft() for _ in range(len(queue))]
        parent_t = next((p for n, p in level if n.val == target), "absent")
        if parent_t != "absent":
            return sorted(n.val for n, p in level if p is not parent_t)
        for n, _ in level:
            for c in (n.left, n.right):
                if c:
                    queue.append((c, n))
    return []

def ans_are_siblings(root, x, y):
    if root is None:
        return False
    if root.left and root.right and {root.left.val, root.right.val} == {x, y}:
        return True
    return ans_are_siblings(root.left, x, y) or ans_are_siblings(root.right, x, y)

def ans_distance(root, x, y):
    def lca(node):
        if node is None or node.val in (x, y):
            return node
        l, r = lca(node.left), lca(node.right)
        if l and r:
            return node
        return l or r
    a = lca(root)
    return ans_depth_of(a, x) + ans_depth_of(a, y)

assert ans_depth_of(build_tree([1, 2, 3, None, 4]), 4) == 2
assert ans_depth_of(build_tree([1, 2, 3]), 9) == -1
assert ans_parent_of(build_tree([1, 2, 3, None, 4]), 4) == 2
assert ans_parent_of(build_tree([1, 2, 3]), 1) is None
assert ans_level_order(build_tree([3, 9, 20, None, None, 15, 7])) == [[3], [9, 20], [15, 7]]
assert ans_cousins_of(build_tree([1, 2, 3, 4, 5, 6, 7]), 4) == [6, 7]
assert ans_cousins_of(build_tree([1, 2, 3]), 2) == []
assert ans_are_siblings(build_tree([1, 2, 3, None, 4]), 2, 3) == True
assert ans_are_siblings(build_tree([1, 2, 3, 4, 5, 6, 7]), 4, 6) == False
assert ans_distance(build_tree([1, 2, 3, 4, 5, 6, 7]), 4, 6) == 4
assert ans_distance(build_tree([1, 2, 3, 4, 5, 6, 7]), 4, 5) == 2
assert ans_distance(build_tree([1, 2, 3, 4, 5, 6, 7]), 1, 7) == 2
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


`next(iterator, default)` lấy phần tử **đầu tiên** của iterator, hoặc trả `default` nếu rỗng — tiện để "tìm phần tử đầu tiên thoả điều kiện".

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isCousins(self, root, x: int, y: int) -> bool:
        queue = deque([root])
        while queue:
            fx = fy = False
            for _ in range(len(queue)):
                node = queue.popleft()
                if node.left and node.right and {node.left.val, node.right.val} == {x, y}:
                    return False                      # anh em ruột
                fx |= node.val == x
                fy |= node.val == y
                if node.left: queue.append(node.left)
                if node.right: queue.append(node.right)
            if fx and fy: return True                 # cùng tầng, khác cha
            if fx or fy: return False                 # khác tầng
        return False
```

**Checklist trước khi submit**
- [ ] Kiểm tra **cả hai** điều kiện: cùng độ sâu **và** khác cha?
- [ ] BFS tách tầng bằng `for _ in range(len(queue))`?
- [ ] Kiểm tra con tồn tại trước khi đọc `.val`?
- [ ] Trường hợp `x` hoặc `y` là root → `False`?
- [ ] Kết quả không phụ thuộc thứ tự `x`, `y`?
- [ ] Nhớ `from collections import deque` (LeetCode đã import sẵn, nhưng nên viết rõ khi chạy ngoài)?